In [ ]:
# ============================================================================
# Cell 1 — Package installation
# ============================================================================
import os, sys, subprocess

def _pip_install(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"pip install failed for {pkgs}")

_pip_install("easyfsl==1.5.0", "gdown")
print("Packages ready.")

In [ ]:
# ============================================================================
# Cell 2 — MiniImageNet download + extraction  (from GeM_ContrastiveLoss)
# ============================================================================
import tarfile, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_FOLDER   = "https://drive.google.com/drive/folders/17a09kkqVivZQFggCw9I_YboJ23tcexNM"
DOWNLOAD_DIR   = WORK_DIR / "miniimagenet_archives"
DATA_DIR       = WORK_DIR / "miniimagenet"
EXPECTED_CLASSES = {"train": 64, "val": 16, "test": 20}

def _split_ready(split):
    d = DATA_DIR / split / split
    return d.is_dir() and sum(1 for p in d.iterdir() if p.is_dir()) == EXPECTED_CLASSES[split]

if all(_split_ready(s) for s in EXPECTED_CLASSES):
    print("MiniImageNet already extracted; skipping download.")
else:
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, 4):
        result = subprocess.run(
            [sys.executable, "-m", "gdown", "--folder", DRIVE_FOLDER, "-O", str(DOWNLOAD_DIR)],
            check=False)
        print("Download exit code:", result.returncode, f"(attempt {attempt}/3)")
        if all(list(DOWNLOAD_DIR.rglob(f"{s}.tar")) for s in EXPECTED_CLASSES):
            break
    print("Downloaded files:")
    for p in DOWNLOAD_DIR.rglob("*"):
        if p.is_file():
            print(f"  {p.name}: {p.stat().st_size / 1e6:.1f} MB")
    for split in ("train", "val", "test"):
        matches = list(DOWNLOAD_DIR.rglob(f"{split}.tar"))
        if not matches:
            raise FileNotFoundError(
                f"{split}.tar not found. Check Drive folder is public, "
                "Internet is enabled, and download quota is not exhausted.")
        archive = matches[0]
        out_dir = DATA_DIR / split
        out_dir.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {archive.name} -> {out_dir} ...")
        with tarfile.open(archive, "r:*") as tar:
            try:
                tar.extractall(out_dir, filter="data")
            except TypeError:
                tar.extractall(out_dir)
        print(f"Finished extracting {split}")

for s in EXPECTED_CLASSES:
    if not _split_ready(s):
        raise RuntimeError(
            f"Split '{s}' not correctly extracted "
            f"(expected {EXPECTED_CLASSES[s]} class folders in {DATA_DIR / s / s}).")
print("Dataset ready.")

In [ ]:
# ============================================================================
# Cell 3 — Imports, configuration, and memory helpers
# ============================================================================
import os, gc, io, time, copy, json, math, random, shutil, hashlib, ctypes
import platform, multiprocessing, contextlib
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names
from easyfsl.samplers import TaskSampler
from tqdm.auto import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

try:
    import psutil
except ImportError:
    psutil = None

try:
    multiprocessing.set_start_method("spawn", force=True)
except RuntimeError:
    pass

# ── ONE-WORD DISTANCE SWITCH ──────────────────────────────────────────────────
# Change "euclidean" to "cosine" to run all four experiments with cosine distance.
DISTANCE_METRIC = "euclidean"

# ── Seeds ─────────────────────────────────────────────────────────────────────
SEED               = 42
TRAIN_EPISODE_SEED = 20263
VAL_EPISODE_SEED   = 20261
TEST_EPISODE_SEED  = 20262

# ── Baseline hyperparameters (unchanged from baseline.ipynb) ──────────────────
IMAGE_SIZE           = 84
MAX_EPOCHS           = 10
N_TRAINING_EPISODES  = 360
N_VALIDATION_TASKS   = 600
N_TEST_TASKS         = 600
TRAIN_WAY            = 30
TEST_WAY             = 5
SHOT                 = 5
QUERY                = 15
FEATURE_CHUNK_SIZE   = 600
ATTENTION_CHUNK_SIZE = 600
COSINE_TEMP          = 10.0
USE_AMP              = True
STAGE_NODES          = ("maxpool", "layer1", "layer2", "layer3", "layer4")
STAGE_WEIGHTS        = (1.0, 1.1, 1.2, 1.3, 1.4)
USE_IMAGENET_PRETRAINED = True
NUM_WORKERS          = 0
gamma_init           = 0.2
weights_lr           = 0.005
backbone_lr          = 1e-4
AMP_ON               = bool(USE_AMP and torch.cuda.is_available())

# ── GeM hyperparameters (from GeM_ContrastiveLoss) ───────────────────────────
GEM_P_INIT  = 3.0
GEM_EPS     = 1e-6
GEM_P_FLOOR = 1e-2
GEM_LR      = 1e-3

# ── EMA hyperparameter (from 5-way-ablation A4) ───────────────────────────────
EMA_BETA = 0.99

# ── Experiment configurations ─────────────────────────────────────────────────
# E0/C0 = baseline, E1/C1 = EMA+LN+GeM, E2/C2 = EMA+GeM, E3/C3 = EMA+LN
PREFIX = "C" if DISTANCE_METRIC.lower() == "cosine" else "E"
EXPERIMENT_CONFIGS = {
    f"{PREFIX}0": dict(description="Original Baseline",     use_ema=False, use_gem=False, use_layernorm=False),
    f"{PREFIX}1": dict(description="EMA + LayerNorm + GeM", use_ema=True,  use_gem=True,  use_layernorm=True),
    f"{PREFIX}2": dict(description="EMA + GeM",             use_ema=True,  use_gem=True,  use_layernorm=False),
    f"{PREFIX}3": dict(description="EMA + LayerNorm",       use_ema=True,  use_gem=False, use_layernorm=True),
}
CONFIG_ORDER = [f"{PREFIX}0", f"{PREFIX}1", f"{PREFIX}2", f"{PREFIX}3"]

# ── Data paths ────────────────────────────────────────────────────────────────
WORK_DIR  = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working").resolve()
DATA_DIR  = WORK_DIR / "miniimagenet"
TRAIN_DIR = str(DATA_DIR / "train" / "train")
VAL_DIR   = str(DATA_DIR / "val"   / "val")
TEST_DIR  = str(DATA_DIR / "test"  / "test")

RESULT_DIR = WORK_DIR / "results"
COMP_DIR   = RESULT_DIR / "comparison_tables"
PARAM_DIR  = RESULT_DIR / "parameter_tracking"
PLOT_DIR   = RESULT_DIR / "plots"
for d in (COMP_DIR, PARAM_DIR, PLOT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── RAM helpers ───────────────────────────────────────────────────────────────
RAM_LIMIT_GB  = 29.0
RAM_WARN_FRAC = 0.80
RAM_FAIL_FRAC = 0.92

def rss_gb():
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 2**20
    except Exception:
        pass
    if psutil is not None:
        return psutil.Process().memory_info().rss / 2**30
    return float("nan")

def free_host_memory():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except OSError:
        pass

def ram_guard(tag=""):
    rss = rss_gb()
    if rss > RAM_WARN_FRAC * RAM_LIMIT_GB:
        free_host_memory(); rss = rss_gb()
        print(f"[RAM] WARNING {tag}: RSS {rss:.1f} GB above {RAM_WARN_FRAC:.0%} of {RAM_LIMIT_GB:.0f} GB limit.")
        if rss > RAM_FAIL_FRAC * RAM_LIMIT_GB:
            raise MemoryError(f"Process RSS {rss:.1f} GB exceeds {RAM_FAIL_FRAC:.0%} of {RAM_LIMIT_GB:.0f} GB limit.")
    return rss

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("WARNING: No GPU detected. Training will be extremely slow on CPU.")

ENV_INFO = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
    "distance_metric": DISTANCE_METRIC,
}
try:
    import torchvision, easyfsl
    ENV_INFO["torchvision"] = torchvision.__version__
    ENV_INFO["easyfsl"] = getattr(easyfsl, "__version__", "unknown")
except Exception:
    pass

PRELOAD_THREADS  = min(16, max(4, (os.cpu_count() or 4) * 2))
resize_transform = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))
IMG_MEAN = (0.485, 0.456, 0.406)
IMG_STD  = (0.229, 0.224, 0.225)

print(f"Device: {device} | GPU: {ENV_INFO['gpu']}")
print(f"DISTANCE_METRIC = '{DISTANCE_METRIC}'  (change to 'cosine' to switch all experiments)")
print(f"Environment: {ENV_INFO}")
print(f"[RAM] start: {rss_gb():.2f} GB (limit {RAM_LIMIT_GB:.0f} GB)")

In [ ]:
# ============================================================================
# Cell 4 — RAM-cached datasets (PreloadedSet from GeM_ContrastiveLoss/Source Code.ipynb)
# ============================================================================

_mean_gpu = torch.tensor(IMG_MEAN, dtype=torch.float32).view(1, 3, 1, 1)
_std_gpu  = torch.tensor(IMG_STD,  dtype=torch.float32).view(1, 3, 1, 1)

def to_dev(images_u8, train=False):
    """uint8 (N,3,H,W) CPU tensor -> normalised float32 CUDA tensor.
    Equivalent to torchvision ToTensor (/255) + Normalize(mean, std);
    with train=True also applies RandomHorizontalFlip(p=0.5) independently per image."""
    x = images_u8.pin_memory().to(device, non_blocking=True).float().div_(255.0)
    if train:
        flip = torch.rand(x.shape[0], device=device) < 0.5
        x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    return x.sub_(_mean_gpu.to(device)).div_(_std_gpu.to(device))


class PreloadedSet:
    """All images of one split decoded + resized ONCE, kept in RAM as uint8 (N,3,H,W).
    Labels and class order from torchvision ImageFolder (identical to the baseline)."""
    def __init__(self, root, name):
        folder = ImageFolder(root)          # no transform; only used for file listing / labels
        self.classes, self.targets = folder.classes, list(folder.targets)
        paths = [pth for pth, _ in folder.samples]
        n = len(paths)
        self.images = torch.empty((n, 3, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.uint8)
        view = self.images.numpy()          # shares memory; threads write their own slices
        def _load(i):
            with Image.open(paths[i]) as im:
                im = resize_transform(im.convert("RGB"))
            view[i] = np.asarray(im, dtype=np.uint8).transpose(2, 0, 1)
        t0 = time.time()
        with ThreadPoolExecutor(max_workers=PRELOAD_THREADS) as pool:
            for _ in tqdm(pool.map(_load, range(n)), total=n,
                          desc=f"Preloading {name}", leave=False):
                pass
        self.labels = torch.tensor(self.targets, dtype=torch.long)
        print(f"{name}: {n} images preloaded in {time.time()-t0:.0f}s "
              f"-> {self.images.numel()/2**20:.0f} MB uint8 in RAM")
    def __len__(self): return len(self.targets)
    def get_labels(self): return self.targets   # EasyFSL TaskSampler compatibility


train_set = PreloadedSet(TRAIN_DIR, "train")
val_set   = PreloadedSet(VAL_DIR,   "val")
test_set  = PreloadedSet(TEST_DIR,  "test")
free_host_memory()
print(f"Classes: train={len(train_set.classes)}, val={len(val_set.classes)}, test={len(test_set.classes)}")
print(f"Images:  train={len(train_set)}, val={len(val_set)}, test={len(test_set)}")
assert len(train_set.classes) >= TRAIN_WAY
assert len(val_set.classes)   >= TEST_WAY
assert len(test_set.classes)  >= TEST_WAY
print(f"[RAM] after caching datasets: {rss_gb():.2f} GB")

In [ ]:
# ============================================================================
# Cell 5 — EpisodeLoader, TaskSampler patch, val/test loaders + smoke test
# ============================================================================

# EasyFSL compatibility patch (verbatim from baseline.ipynb)
def _patched_task_sampler_init(self, dataset, n_way, n_shot, n_query, n_tasks):
    Sampler.__init__(self)
    self.n_way, self.n_shot, self.n_query, self.n_tasks = n_way, n_shot, n_query, n_tasks
    self.items_per_label = defaultdict(list)
    for item, label in enumerate(dataset.get_labels()):
        self.items_per_label[int(label)].append(item)
    if len(self.items_per_label) < n_way:
        raise ValueError(
            f"Requested {n_way}-way episodes but dataset has only {len(self.items_per_label)} classes.")
    required = n_shot + n_query
    too_small = [l for l, items in self.items_per_label.items() if len(items) < required]
    if too_small:
        raise ValueError(f"Classes have fewer than {required} images: {too_small[:5]}")

TaskSampler.__init__ = _patched_task_sampler_init


class EpisodeLoader:
    """Fast collation from PreloadedSet (indices from unchanged EasyFSL TaskSampler).
    Yields (support_u8, support_labels, query_u8, query_labels, class_ids).
    Images are uint8 CPU tensors — call to_dev() for GPU preprocessing."""
    def __init__(self, store, n_way, n_tasks, shot=SHOT, query=QUERY):
        self.store   = store
        self.n_way   = n_way
        self.shot    = shot
        self.query   = query
        self.sampler = TaskSampler(store, n_way=n_way, n_shot=shot, n_query=query, n_tasks=n_tasks)
    def __len__(self): return len(self.sampler)
    def collate(self, idx):
        idx_t     = torch.as_tensor(idx, dtype=torch.long)
        lab       = self.store.labels[idx_t].tolist()
        class_ids = list({x for x in lab})
        remap     = {c: i for i, c in enumerate(class_ids)}
        labels    = torch.tensor([remap[c] for c in lab]).reshape(self.n_way, self.shot + self.query)
        imgs      = self.store.images[idx_t].reshape(
            self.n_way, self.shot + self.query, 3, IMAGE_SIZE, IMAGE_SIZE)
        return (imgs[:, :self.shot].reshape(-1, 3, IMAGE_SIZE, IMAGE_SIZE),
                labels[:, :self.shot].flatten(),
                imgs[:, self.shot:].reshape(-1, 3, IMAGE_SIZE, IMAGE_SIZE),
                labels[:, self.shot:].flatten(),
                class_ids)
    def __iter__(self):
        for idx in self.sampler:
            yield self.collate(idx)


def make_loader(dataset, n_way, n_tasks, shot=SHOT, query=QUERY):
    loader = EpisodeLoader(dataset, n_way, n_tasks, shot, query)
    assert len(loader) == n_tasks, f"Loader has {len(loader)} tasks, expected {n_tasks}."
    return loader


# Fixed val and test loaders (same episodes for all configurations)
set_seed(VAL_EPISODE_SEED)
val_loader  = make_loader(val_set,  TEST_WAY, N_VALIDATION_TASKS)
set_seed(TEST_EPISODE_SEED)
test_loader = make_loader(test_set, TEST_WAY, N_TEST_TASKS)
print(f"val={len(val_loader)} tasks, test={len(test_loader)} tasks")

def check_episode(loader, way, shot=SHOT):
    s_img, s_lab, q_img, q_lab, _ = next(iter(loader))
    assert s_img.dtype == torch.uint8 and q_img.dtype == torch.uint8
    assert s_img.shape == (way * shot, 3, IMAGE_SIZE, IMAGE_SIZE), s_img.shape
    assert q_img.shape == (way * QUERY, 3, IMAGE_SIZE, IMAGE_SIZE), q_img.shape
    assert len(s_lab) == way * shot and len(q_lab) == way * QUERY
    assert len(set(s_lab.tolist())) == way
    print(f"  {way}-way {shot}-shot episode OK: {tuple(s_img.shape)} / {tuple(q_img.shape)}")

t0 = time.time()
check_episode(val_loader,  TEST_WAY)
check_episode(test_loader, TEST_WAY)
print(f"Smoke test done in {time.time()-t0:.2f}s")
print("TaskSampler patched; EpisodeLoader ready.")

In [ ]:
# ============================================================================
# Cell 6 — Model: Self_Attn + signed GeM + PrototypicalNetworks
#           (baseline code verbatim + opt-in use_gem and use_layernorm flags)
# ============================================================================

class Self_Attn(nn.Module):
    """Self-attention module (verbatim from baseline.ipynb)."""
    def __init__(self, in_dim):
        super().__init__()
        if in_dim < 8:
            raise ValueError(f"in_dim must be >= 8, got {in_dim}")
        self.query_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.key_conv   = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.value_conv = nn.Conv2d(in_dim, in_dim,      kernel_size=1)

    def forward(self, x, gamma):
        b, c, h, w = x.shape
        n = h * w
        q = self.query_conv(x).reshape(b, -1, n).transpose(1, 2)
        k = self.key_conv(x).reshape(b, -1, n)
        attention = torch.softmax(torch.bmm(q, k), dim=-1)
        v = self.value_conv(x).reshape(b, c, n)
        out = torch.bmm(v, attention.transpose(1, 2)).reshape(b, c, h, w)
        return gamma * out + x


# Signed GeM pooling (from GeM_ContrastiveLoss/Source Code - Initial.ipynb Cell 6)
def _gem_nonneg(t, p, eps):
    """Plain GeM of a non-negative map (B,C,H,W) via log-sum-exp, float32, numerically stable."""
    logt = t.clamp(min=eps).log().flatten(2)
    return torch.exp((torch.logsumexp(p * logt, dim=-1) - math.log(logt.shape[-1])) / p)

def gem_pool(x, p, eps=GEM_EPS):
    """Signed GeM for possibly-negative activations (attention residual).
    Splits y = y+ - y-  =>  out = GeM_p(y+) - GeM_p(y-).
    For p=1 this equals global average pooling exactly."""
    with torch.autocast(device_type="cuda", enabled=False):
        xf, pf = x.float(), p.float()
        out = _gem_nonneg(xf, pf, eps) - _gem_nonneg(-xf, pf, eps)
    return out.to(x.dtype)


class PrototypicalNetworks(nn.Module):
    """Baseline multi-scale prototypical network with optional extensions:
      use_gem=True       -> signed GeM replaces AdaptiveAvgPool2d in _attend_and_pool
      use_layernorm=True -> nn.LayerNorm on pooled embeddings before prototypes/distance
    When both flags are False this is identical to the original baseline.ipynb model."""

    def __init__(self, backbone, image_size,
                 stage_nodes=STAGE_NODES, weight=STAGE_WEIGHTS,
                 weightlearnable=True, gammalearnable=True,
                 gamma1=0.2, gamma2=0.2,
                 feature_chunk_size=600, attention_chunk_size=600,
                 distance_metric="euclidean", cosine_temp=10.0,
                 use_gem=False, use_layernorm=False, ln_shared=True):
        super().__init__()
        if len(weight) != len(stage_nodes):
            raise ValueError(f"{len(weight)} weights given for {len(stage_nodes)} stages.")
        self.weightlearnable      = weightlearnable
        self.gammalearnable       = gammalearnable
        self.feature_chunk_size   = feature_chunk_size
        self.attention_chunk_size = attention_chunk_size
        self.n_stages             = len(stage_nodes)
        self.distance_metric      = distance_metric
        self.use_gem              = use_gem
        self.use_layernorm        = use_layernorm
        self.ln_shared            = ln_shared

        # Baseline learnable parameters
        self.cosine_temp  = nn.Parameter(torch.tensor(float(cosine_temp)), requires_grad=True)
        self.stage_weights = nn.Parameter(
            torch.tensor(list(weight), dtype=torch.float32), requires_grad=weightlearnable)
        self.gamma1_l = nn.Parameter(torch.tensor(float(gamma1)), requires_grad=gammalearnable)
        self.gamma2_l = nn.Parameter(torch.tensor(float(gamma2)), requires_grad=gammalearnable)

        # Build feature extractor
        graph_nodes, _ = get_graph_node_names(backbone)
        return_nodes = {}
        for i, name in enumerate(stage_nodes):
            matches = [n for n in graph_nodes if n == name or n.startswith(name + ".")]
            if not matches:
                raise ValueError(f"Stage {name!r} not found in backbone graph.")
            return_nodes[matches[-1]] = f"stage_{i}"
        print("Stage nodes:", return_nodes)
        self.feature_extractor = create_feature_extractor(backbone, return_nodes=return_nodes)

        was_training = self.feature_extractor.training
        self.feature_extractor.eval()
        with torch.no_grad():
            dummy = self.feature_extractor(torch.zeros(1, 3, image_size, image_size))
        self.feature_extractor.train(was_training)
        channels = [dummy[f"stage_{i}"].shape[1] for i in range(self.n_stages)]
        print("Selected feature channels:", channels,
              "| spatial sizes:", [tuple(dummy[f"stage_{i}"].shape[2:]) for i in range(self.n_stages)])

        self.attention_modules = nn.ModuleList([Self_Attn(c) for c in channels])
        self.global_avg_pool   = nn.AdaptiveAvgPool2d(1)   # baseline GAP (used when use_gem=False)

        # GeM: one learnable exponent per stage (init so p == GEM_P_INIT exactly)
        if use_gem:
            raw0 = math.log(math.expm1(GEM_P_INIT - GEM_P_FLOOR))
            self.gem_raw = nn.Parameter(
                torch.full((self.n_stages,), raw0, dtype=torch.float32))

        # LayerNorm: one module per stage (from LayerNorm_LearnableTemp notebook)
        if use_layernorm:
            self.stage_norms_support = nn.ModuleList([nn.LayerNorm(c) for c in channels])
            if not ln_shared:
                self.stage_norms_query = nn.ModuleList([nn.LayerNorm(c) for c in channels])

    @property
    def gem_p(self):
        """Strictly positive GeM exponents, one per stage."""
        return F.softplus(self.gem_raw) + GEM_P_FLOOR

    def _extract_in_chunks(self, images):
        if images.size(0) <= self.feature_chunk_size:
            return self.feature_extractor(images)
        collected = None
        for chunk in images.split(self.feature_chunk_size, dim=0):
            feats = self.feature_extractor(chunk)
            if collected is None:
                collected = {k: [v] for k, v in feats.items()}
            else:
                for k, v in feats.items(): collected[k].append(v)
        return {k: torch.cat(vs, dim=0) for k, vs in collected.items()}

    def _attend_and_pool(self, features, stage_idx, gamma):
        module = self.attention_modules[stage_idx]
        if self.use_gem:
            # Signed GeM replaces GAP (from GeM_ContrastiveLoss, _gem_stage pattern)
            p = self.gem_p[stage_idx]
            outs = []
            for chunk in features.split(self.attention_chunk_size, dim=0):
                a = module(chunk, gamma)    # y = gamma*attn + x  (can be negative)
                outs.append(gem_pool(a, p))
            return outs[0] if len(outs) == 1 else torch.cat(outs, dim=0)
        else:
            # Original baseline GAP path (verbatim from baseline.ipynb)
            if features.size(0) <= self.attention_chunk_size:
                return self.global_avg_pool(module(features, gamma)).flatten(1)
            pooled = []
            for chunk in features.split(self.attention_chunk_size, dim=0):
                pooled.append(self.global_avg_pool(module(chunk, gamma)).flatten(1))
            return torch.cat(pooled, dim=0)

    def _stage_distance(self, support_features, query_features, support_labels, stage_idx):
        z_support = self._attend_and_pool(support_features, stage_idx, self.gamma1_l)
        z_query   = self._attend_and_pool(query_features,   stage_idx, self.gamma2_l)

        # LayerNorm AFTER pool, BEFORE prototype computation (from LayerNorm_LearnableTemp notebook)
        if self.use_layernorm:
            z_support = self.stage_norms_support[stage_idx](z_support)
            q_norms   = self.stage_norms_support if self.ln_shared else self.stage_norms_query
            z_query   = q_norms[stage_idx](z_query)

        labels     = torch.unique(support_labels, sorted=True)
        prototypes = torch.stack([z_support[support_labels == l].mean(0) for l in labels])

        # Distance computation (verbatim from baseline.ipynb _stage_distance)
        if self.distance_metric == "euclidean":
            return torch.cdist(z_query, prototypes, p=2)
        elif self.distance_metric == "cosine":
            z_q_n   = F.normalize(z_query,     p=2, dim=-1)
            p_n     = F.normalize(prototypes,   p=2, dim=-1)
            cos_sim = torch.mm(z_q_n, p_n.t())
            return self.cosine_temp * (1.0 - cos_sim)
        elif self.distance_metric == "hybrid":
            euc   = torch.cdist(z_query, prototypes, p=2)
            z_q_n = F.normalize(z_query,   p=2, dim=-1)
            p_n   = F.normalize(prototypes, p=2, dim=-1)
            cos   = self.cosine_temp * (1.0 - torch.mm(z_q_n, p_n.t()))
            return 0.5 * euc + 0.5 * cos
        else:
            raise ValueError(f"Unknown distance_metric: {self.distance_metric}")

    def forward(self, support_images, support_labels, query_images):
        support_feats = self._extract_in_chunks(support_images)
        query_feats   = self._extract_in_chunks(query_images)
        distances = torch.stack([
            self._stage_distance(support_feats[f"stage_{i}"],
                                 query_feats[f"stage_{i}"],
                                 support_labels, i)
            for i in range(self.n_stages)
        ])
        return -(self.stage_weights.view(-1, 1, 1) * distances).sum(dim=0)

print("Model class defined.")

In [ ]:
# ============================================================================
# Cell 7 — ModelEMA + evaluate_full + pretrained cache + model/optimizer builders
# ============================================================================

class ModelEMA:
    """EMA of the complete model state (parameters AND BN running statistics).
    From 5-way-ablation A4, verbatim."""
    def __init__(self, model, beta):
        self.beta   = beta
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        ema_sd, live_sd = self.module.state_dict(), model.state_dict()
        fkeys = [k for k, v in ema_sd.items() if v.is_floating_point()]
        okeys = [k for k in ema_sd if k not in set(fkeys)]
        self.ema_f  = [ema_sd[k]  for k in fkeys]
        self.live_f = [live_sd[k] for k in fkeys]
        self.ema_o  = [ema_sd[k]  for k in okeys]
        self.live_o = [live_sd[k] for k in okeys]

    @torch.no_grad()
    def update(self, model=None):
        torch._foreach_mul_(self.ema_f, self.beta)
        torch._foreach_add_(self.ema_f, self.live_f, alpha=1.0 - self.beta)
        for e, l in zip(self.ema_o, self.live_o):
            e.copy_(l)


class fixed_rng:
    """Draw evaluation episodes from a fixed seed, then restore the previous RNG state."""
    def __init__(self, seed): self.seed = seed
    def __enter__(self):
        self.py = random.getstate(); self.np = np.random.get_state()
        self.t  = torch.get_rng_state()
        self.cu = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        random.seed(self.seed); np.random.seed(self.seed); torch.manual_seed(self.seed)
    def __exit__(self, *exc):
        random.setstate(self.py); np.random.set_state(self.np); torch.set_rng_state(self.t)
        if self.cu is not None: torch.cuda.set_rng_state_all(self.cu)
        return False


@torch.no_grad()
def evaluate_full(data_loader, model, seed, desc="Evaluating", timed=False):
    """Baseline evaluate + CE loss + per-episode accuracy + optional timing."""
    model.eval()
    task_acc, task_loss, ep_time = [], [], []
    if timed:
        s0, l0, q0, _, _ = next(iter(data_loader))
        S0, Q0, L0 = to_dev(s0), to_dev(q0), l0.to(device)
        for _ in range(3): model(S0, L0, Q0)
        if device.type == "cuda": torch.cuda.synchronize()
    with fixed_rng(seed):
        for support_images, support_labels, query_images, query_labels, _ in                 tqdm(data_loader, total=len(data_loader), desc=desc, leave=False):
            s_img  = to_dev(support_images)
            q_img  = to_dev(query_images)
            s_lab  = support_labels.to(device)
            labels = query_labels.to(device)
            if timed:
                if device.type == "cuda": torch.cuda.synchronize()
                t0 = time.perf_counter()
            scores = model(s_img, s_lab, q_img)
            if timed:
                if device.type == "cuda": torch.cuda.synchronize()
                ep_time.append(time.perf_counter() - t0)
            task_loss.append(F.cross_entropy(scores.float(), labels).item())
            task_acc.append((scores.argmax(dim=1) == labels).float().mean().item())
    if len(task_acc) != len(data_loader):
        raise RuntimeError(f"Evaluated {len(task_acc)} tasks but loader promised {len(data_loader)}.")
    task_acc = 100.0 * np.array(task_acc)
    return dict(acc=float(task_acc.mean()), loss=float(np.mean(task_loss)),
                task_acc=task_acc, per_episode_time=ep_time)


# Pretrained backbone cached once in RAM (all 4 configs start from this exact state)
def _load_pretrained_state():
    if not USE_IMAGENET_PRETRAINED:
        return None
    sd = resnet18(weights=ResNet18_Weights.DEFAULT).state_dict()
    return {k: v.detach().clone() for k, v in sd.items()}

PRETRAINED_STATE = _load_pretrained_state()
print("Pretrained backbone:", "ENABLED" if PRETRAINED_STATE is not None else "DISABLED (random init)")


def build_model(key, verbose=False):
    """Build a fresh model from PRETRAINED_STATE for the given config key.
    set_seed(SEED) ensures identical random init of attention modules."""
    cfg = EXPERIMENT_CONFIGS[key]
    set_seed(SEED)
    backbone = resnet18(weights=None)
    if PRETRAINED_STATE is not None:
        backbone.load_state_dict(PRETRAINED_STATE)
    backbone.fc = nn.Identity()
    sink = contextlib.nullcontext() if verbose else contextlib.redirect_stdout(io.StringIO())
    with sink:
        model = PrototypicalNetworks(
            backbone=backbone, image_size=IMAGE_SIZE,
            stage_nodes=STAGE_NODES, weight=STAGE_WEIGHTS,
            weightlearnable=True, gammalearnable=True,
            gamma1=gamma_init, gamma2=gamma_init,
            feature_chunk_size=FEATURE_CHUNK_SIZE,
            attention_chunk_size=ATTENTION_CHUNK_SIZE,
            distance_metric=DISTANCE_METRIC, cosine_temp=COSINE_TEMP,
            use_gem=cfg["use_gem"],
            use_layernorm=cfg["use_layernorm"],
        )
    return model.to(device)


def build_optimizer(model, cfg):
    """Baseline Adam param groups + GeM exponent group if use_gem.
    LayerNorm params are in regular_params (backbone_lr)."""
    special = [model.stage_weights, model.gamma1_l, model.gamma2_l, model.cosine_temp]
    if cfg["use_gem"]:
        special.append(model.gem_raw)
    special_ids    = {id(p) for p in special}
    regular_params = [p for p in model.parameters() if id(p) not in special_ids]
    param_groups = [
        {"params": regular_params,       "lr": backbone_lr},
        {"params": [model.cosine_temp],  "lr": weights_lr},
    ]
    if model.weightlearnable:
        param_groups.append({"params": [model.stage_weights], "lr": weights_lr})
    if model.gammalearnable:
        param_groups.append({"params": [model.gamma1_l, model.gamma2_l], "lr": weights_lr})
    if cfg["use_gem"]:
        param_groups.append({"params": [model.gem_raw], "lr": GEM_LR})
    return optim.Adam(param_groups)


def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def snapshot_params(model):
    """Lightweight numerical snapshot of all tracked learnable scalars."""
    d = {f"stage_w{i+1}": v for i, v in
         enumerate(model.stage_weights.detach().cpu().tolist())}
    d["gamma1"] = float(model.gamma1_l.detach().cpu())
    d["gamma2"] = float(model.gamma2_l.detach().cpu())
    if hasattr(model, "gem_raw"):
        for i, v in enumerate(model.gem_p.detach().cpu().tolist()):
            d[f"gem_p{i+1}"] = v
    return d


print("Helpers defined.")

In [ ]:
# ============================================================================
# Cell 8 — Lightweight smoke tests
# ============================================================================
print("Running smoke tests...")
t0 = time.time()

for key, cfg in EXPERIMENT_CONFIGS.items():
    m = build_model(key)
    n_total, n_trainable = count_params(m)
    s  = torch.randn(TEST_WAY * SHOT,  3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    sl = torch.arange(TEST_WAY, device=device).repeat_interleave(SHOT)
    q  = torch.randn(TEST_WAY * QUERY, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    ql = torch.arange(TEST_WAY, device=device).repeat_interleave(QUERY)
    # Forward pass (AMP)
    with torch.amp.autocast("cuda", enabled=AMP_ON):
        logits = m(s, sl, q)
    assert logits.shape == (TEST_WAY * QUERY, TEST_WAY), logits.shape
    # Backward pass
    F.cross_entropy(logits.float(), ql).backward()
    # EMA update
    if cfg["use_ema"]:
        ema = ModelEMA(m, EMA_BETA); ema.update()
    # Both distance metrics
    for dm in ("euclidean", "cosine"):
        m.distance_metric = dm
        with torch.no_grad(): m(s, sl, q)
    m.distance_metric = DISTANCE_METRIC
    print(f"  [{key}] {cfg['description']:25s}  total={n_total:,}  trainable={n_trainable:,}  OK")
    del m
    free_gpu()

print(f"All smoke tests passed in {time.time()-t0:.1f}s")

In [ ]:
# ============================================================================
# Cell 9 — run_experiment(key): complete training + evaluation for one config
# ============================================================================

RESULTS = {}   # filled by Cell 10

def run_experiment(key):
    cfg = EXPERIMENT_CONFIGS[key]
    print("=" * 72)
    print(f"  CONFIG [{key}]: {cfg['description']}")
    print(f"  use_ema={cfg['use_ema']}  use_gem={cfg['use_gem']}  use_layernorm={cfg['use_layernorm']}")
    print(f"  DISTANCE_METRIC = '{DISTANCE_METRIC}'")
    print("=" * 72)
    free_gpu()
    wall0 = time.time()

    # ── Build fresh model + optimizer ────────────────────────────────────────
    model     = build_model(key, verbose=(key == CONFIG_ORDER[0]))
    optimizer = build_optimizer(model, cfg)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler("cuda", enabled=AMP_ON)
    trainable = [p for p in model.parameters() if p.requires_grad]
    criterion = nn.CrossEntropyLoss()
    n_total, n_trainable = count_params(model)
    print(f"  Parameters: total={n_total:,}  trainable={n_trainable:,}")

    # ── EMA (A4): wraps model; eval uses EMA weights ──────────────────────────
    ema        = ModelEMA(model, EMA_BETA) if cfg["use_ema"] else None
    eval_model = ema.module if ema is not None else model

    trajectory = [dict(step=0, epoch=0, **snapshot_params(model))]
    best_val_acc, best_epoch, best_state = -1.0, 0, None
    epoch_log        = []
    train_time_total = 0.0
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    # ── Epoch loop ────────────────────────────────────────────────────────────
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        # All configs see identical training episodes (seeded per-epoch)
        set_seed(TRAIN_EPISODE_SEED + (epoch - 1))
        train_loader_epoch = make_loader(train_set, TRAIN_WAY, N_TRAINING_EPISODES)

        loss_sum   = torch.zeros((), dtype=torch.float64, device=device)
        acc_sum    = loss_sum.clone()
        n_episodes = 0
        lr_now     = optimizer.param_groups[0]["lr"]

        if device.type == "cuda": torch.cuda.synchronize()
        t_train0 = time.time()

        progress = tqdm(train_loader_epoch, total=len(train_loader_epoch),
                        desc=f"[{key}] Epoch {epoch}/{MAX_EPOCHS}", leave=False)

        for support_images, support_labels, query_images, query_labels, _ in progress:
            optimizer.zero_grad(set_to_none=True)
            sl   = support_labels.to(device, non_blocking=True)
            ql   = query_labels.to(device,   non_blocking=True)
            S_in = to_dev(support_images, train=True)
            Q_in = to_dev(query_images,   train=True)

            with torch.amp.autocast("cuda", enabled=AMP_ON):
                scores = model(S_in, sl, Q_in)
                ce     = criterion(scores, ql)

            scaler.scale(ce).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()

            if ema is not None:
                ema.update()

            loss_sum += ce.detach().double()
            acc_sum  += (scores.detach().argmax(dim=1) == ql).float().mean().double()
            n_episodes += 1

        if device.type == "cuda": torch.cuda.synchronize()
        train_time = time.time() - t_train0
        train_time_total += train_time

        if n_episodes != N_TRAINING_EPISODES:
            raise RuntimeError(f"Ran {n_episodes} episodes, expected {N_TRAINING_EPISODES}.")

        scheduler.step()
        train_loss = float(loss_sum.item() / n_episodes)
        train_acc  = 100.0 * float(acc_sum.item() / n_episodes)

        # Snapshot learnable params after this epoch
        trajectory.append(dict(
            step=epoch * N_TRAINING_EPISODES, epoch=epoch, **snapshot_params(model)))

        # ── Validation (always uses EMA weights when EMA is enabled) ─────────
        t_val0 = time.time()
        val    = evaluate_full(val_loader, eval_model, VAL_EPISODE_SEED,
                               desc=f"[{key}] Val epoch {epoch}")
        val_time = time.time() - t_val0

        row = dict(
            config=key, epoch=epoch,
            train_acc=train_acc, train_loss=train_loss,
            val_acc=val["acc"],  val_loss=val["loss"],
            train_time_s=train_time, val_time_s=val_time, lr=lr_now,
            gpu_peak_mb=(torch.cuda.max_memory_allocated()/2**20
                         if device.type == "cuda" else 0.0),
            **snapshot_params(model),
        )
        epoch_log.append(row)

        # ── Epoch print (exact required format) ──────────────────────────────
        print(f"Epoch {epoch}: train loss(CE)={train_loss:.4f}, train acc={train_acc:.2f}%, "
              f"val loss={val['loss']:.4f}, val accuracy={val['acc']:.2f}%, "
              f"time={(train_time+val_time)/60:.1f} min, "
              f"RAM={ram_guard(f'{key} epoch {epoch}'):.1f} GB")

        if val["acc"] > best_val_acc:
            best_val_acc, best_epoch = val["acc"], epoch
            best_state = {k: v.detach().cpu().clone()
                          for k, v in eval_model.state_dict().items()}

    peak_train_gpu_mb = (torch.cuda.max_memory_allocated()/2**20
                         if device.type == "cuda" else 0.0)
    print(f"[{key}] Best validation: {best_val_acc:.2f}% @ epoch {best_epoch}")

    # ── Release training objects; rebuild fresh model for test ────────────────
    params_end = snapshot_params(model)
    del model, optimizer, scheduler, scaler, ema, eval_model, trainable
    del train_loader_epoch, progress, loss_sum, acc_sum
    free_gpu()

    infer_model = build_model(key)
    infer_model.load_state_dict(best_state)
    del best_state
    free_gpu()  # resets peak stats so inference memory is measured separately

    # ── Final test (5-way 5-shot, timed) ─────────────────────────────────────
    test = evaluate_full(test_loader, infer_model, TEST_EPISODE_SEED,
                         desc=f"[{key}] Test 5-way 5-shot", timed=True)
    peak_infer_gpu_mb = (torch.cuda.max_memory_allocated()/2**20
                         if device.type == "cuda" else 0.0)
    params_best = snapshot_params(infer_model)

    t_arr = np.array(test["per_episode_time"]) * 1000.0
    infer_ms_mean = float(t_arr.mean()) if len(t_arr) > 0 else float("nan")

    # ── Required final output format (from GeM_ContrastiveLoss/Source Code.ipynb) ──
    print("=" * 48)
    print(f"[{key}] FINAL TEST ACCURACY 5-way 5-shot: {test['acc']:.2f}%")
    print("=" * 48)
    sw = [round(params_best[f"stage_w{i+1}"], 4) for i in range(5)]
    g1 = round(params_best["gamma1"], 4)
    g2 = round(params_best["gamma2"], 4)
    print(f"Stage weights (tested model): {sw}")
    print(f"gamma1/gamma2 (tested model): {g1} {g2}")
    if cfg["use_gem"]:
        gp = [round(params_best[f"gem_p{i+1}"], 4) for i in range(5)]
        print(f"GeM p (tested model): {gp}")
    print(f"[RAM] {key} done: {rss_gb():.2f} GB")

    del infer_model
    free_gpu()

    return dict(
        key=key, cfg=cfg,
        epoch_log=epoch_log, trajectory=trajectory,
        params_end_of_training=params_end, params_best_checkpoint=params_best,
        best_epoch=best_epoch, best_val_acc=best_val_acc,
        test_acc=test["acc"],  test_loss=test["loss"],
        per_episode_acc=test["task_acc"].tolist(),
        train_time_total_s=train_time_total,
        peak_train_gpu_mb=peak_train_gpu_mb,
        peak_infer_gpu_mb=peak_infer_gpu_mb,
        infer_ms_per_episode=infer_ms_mean,
        n_total=n_total, n_trainable=n_trainable,
        wall_time_s=time.time() - wall0,
    )

In [ ]:
# ============================================================================
# Cell 10 — Run all four experiments sequentially
# ============================================================================
RESULTS = {}
t_all = time.time()

for key in CONFIG_ORDER:
    RESULTS[key] = run_experiment(key)
    free_gpu()
    ram_guard(f"after {key}")
    print()

print(f"All {len(CONFIG_ORDER)} experiments finished in {(time.time()-t_all)/60:.1f} min")
print()
for key in CONFIG_ORDER:
    r = RESULTS[key]
    print(f"  [{key}] {EXPERIMENT_CONFIGS[key]['description']:25s}  "
          f"test={r['test_acc']:.2f}%  best_val={r['best_val_acc']:.2f}%  "
          f"epoch={r['best_epoch']}")

In [ ]:
# ============================================================================
# Cell 11 — CSV outputs
# ============================================================================

# 1. Main comparison table
base_key = CONFIG_ORDER[0]
base_ep  = np.array(RESULTS[base_key]["per_episode_acc"])
main_rows = []
for key in CONFIG_ORDER:
    r  = RESULTS[key]
    ep = np.array(r["per_episode_acc"])
    std = float(ep.std(ddof=1))
    ci  = float(1.96 * std / math.sqrt(len(ep)))
    d   = ep - base_ep
    delta_mean = float(d.mean())
    delta_ci   = float(1.96 * d.std(ddof=1) / math.sqrt(len(d))) if key != base_key else 0.0
    main_rows.append(dict(
        experiment=key,
        description=EXPERIMENT_CONFIGS[key]["description"],
        distance_metric=DISTANCE_METRIC,
        test_acc_5way5shot=r["test_acc"],
        test_acc_ci95=ci,
        test_acc_std=std,
        delta_vs_baseline=delta_mean,
        delta_ci95_paired=delta_ci,
        best_val_acc=r["best_val_acc"],
        final_val_acc=r["epoch_log"][-1]["val_acc"],
        best_epoch=r["best_epoch"],
        train_time_min=r["train_time_total_s"] / 60,
        wall_time_min=r["wall_time_s"] / 60,
        n_total_params=r["n_total"],
        n_trainable_params=r["n_trainable"],
        peak_train_gpu_mb=r["peak_train_gpu_mb"],
        peak_infer_gpu_mb=r["peak_infer_gpu_mb"],
        infer_ms_per_episode=r["infer_ms_per_episode"],
    ))
df_main = pd.DataFrame(main_rows)
df_main.to_csv(COMP_DIR / "main_comparison.csv", index=False, float_format="%.6g")
print("Saved: main_comparison.csv")

# 2. Computational efficiency
eff_rows = []
for key in CONFIG_ORDER:
    r = RESULTS[key]
    eff_rows.append(dict(
        experiment=key,
        description=EXPERIMENT_CONFIGS[key]["description"],
        train_time_total_s=r["train_time_total_s"],
        train_time_per_epoch_s=r["train_time_total_s"] / MAX_EPOCHS,
        infer_ms_per_episode=r["infer_ms_per_episode"],
        n_total_params=r["n_total"],
        n_trainable_params=r["n_trainable"],
        peak_train_gpu_mb=r["peak_train_gpu_mb"],
        peak_infer_gpu_mb=r["peak_infer_gpu_mb"],
        process_rss_gb=rss_gb(),
    ))
pd.DataFrame(eff_rows).to_csv(COMP_DIR / "computational_efficiency.csv", index=False, float_format="%.6g")
print("Saved: computational_efficiency.csv")

# 3. Improvement over baseline
base_acc = RESULTS[base_key]["test_acc"]
imp_rows = []
for key in CONFIG_ORDER:
    r = RESULTS[key]
    imp_rows.append(dict(
        experiment=key, description=EXPERIMENT_CONFIGS[key]["description"],
        test_acc=r["test_acc"],
        abs_improvement=r["test_acc"] - base_acc,
        rel_improvement_pct=100.0 * (r["test_acc"] - base_acc) / base_acc if base_acc else 0.0,
    ))
pd.DataFrame(imp_rows).to_csv(COMP_DIR / "improvement_over_baseline.csv", index=False, float_format="%.6g")
print("Saved: improvement_over_baseline.csv")

# 4. Per-epoch metrics
epoch_rows = []
for key in CONFIG_ORDER:
    epoch_rows.extend(RESULTS[key]["epoch_log"])
pd.DataFrame(epoch_rows).to_csv(COMP_DIR / "per_epoch_metrics.csv", index=False, float_format="%.6g")
print("Saved: per_epoch_metrics.csv")

# 5a. Parameter trajectories
traj_rows = []
for key in CONFIG_ORDER:
    for snap in RESULTS[key]["trajectory"]:
        traj_rows.append(dict(experiment=key, **snap))
pd.DataFrame(traj_rows).to_csv(PARAM_DIR / "parameter_trajectories.csv", index=False, float_format="%.6g")
print("Saved: parameter_trajectories.csv")

# 5b. Final parameters
final_rows = []
for key in CONFIG_ORDER:
    r = RESULTS[key]
    final_rows.append(dict(experiment=key, source="best_val_checkpoint", **r["params_best_checkpoint"]))
    final_rows.append(dict(experiment=key, source="end_of_training_ep10", **r["params_end_of_training"]))
pd.DataFrame(final_rows).to_csv(PARAM_DIR / "final_parameters.csv", index=False, float_format="%.6g")
print("Saved: final_parameters.csv")

# 6. Experiment config
config_rows = [
    {"key": "distance_metric", "value": DISTANCE_METRIC},
    {"key": "seed", "value": SEED},
    {"key": "train_episode_seed", "value": TRAIN_EPISODE_SEED},
    {"key": "val_episode_seed",   "value": VAL_EPISODE_SEED},
    {"key": "test_episode_seed",  "value": TEST_EPISODE_SEED},
    {"key": "max_epochs",         "value": MAX_EPOCHS},
    {"key": "n_training_episodes","value": N_TRAINING_EPISODES},
    {"key": "n_val_tasks",        "value": N_VALIDATION_TASKS},
    {"key": "n_test_tasks",       "value": N_TEST_TASKS},
    {"key": "train_way",          "value": TRAIN_WAY},
    {"key": "test_way",           "value": TEST_WAY},
    {"key": "shot",               "value": SHOT},
    {"key": "query",              "value": QUERY},
    {"key": "image_size",         "value": IMAGE_SIZE},
    {"key": "backbone_lr",        "value": backbone_lr},
    {"key": "weights_lr",         "value": weights_lr},
    {"key": "gem_lr",             "value": GEM_LR},
    {"key": "ema_beta",           "value": EMA_BETA},
    {"key": "gem_p_init",         "value": GEM_P_INIT},
    {"key": "gem_eps",            "value": GEM_EPS},
    {"key": "gem_p_floor",        "value": GEM_P_FLOOR},
    {"key": "use_amp",            "value": USE_AMP},
    {"key": "cosine_temp_init",   "value": COSINE_TEMP},
    {"key": "stage_weights_init", "value": str(STAGE_WEIGHTS)},
    {"key": "gamma_init",         "value": gamma_init},
    {"key": "use_imagenet_pretrained", "value": USE_IMAGENET_PRETRAINED},
] + [{"key": f"env_{k}", "value": str(v)} for k, v in ENV_INFO.items()]   + [{"key": f"config_{k}", "value": str(v)} for k, v in EXPERIMENT_CONFIGS.items()]
pd.DataFrame(config_rows).to_csv(COMP_DIR / "experiment_config.csv", index=False)
print("Saved: experiment_config.csv")
print("\nAll 6 CSV files saved.")

In [ ]:
# ============================================================================
# Cell 12 — Plots (high-resolution PNG)
# ============================================================================
PALETTE = ["#4c78a8", "#f58518", "#54a24b", "#e45756", "#72b7b2"]
COLORS  = {k: PALETTE[i] for i, k in enumerate(CONFIG_ORDER)}
LABELS  = {k: f"[{k}] {EXPERIMENT_CONFIGS[k]['description']}" for k in CONFIG_ORDER}
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})

def _save(fig, fname):
    fig.tight_layout()
    fig.savefig(PLOT_DIR / fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {fname}")

def line_plot(col, title, ylabel, fname):
    fig, ax = plt.subplots(figsize=(9, 5))
    for k in CONFIG_ORDER:
        d = [r[col] for r in RESULTS[k]["epoch_log"]]
        ax.plot(range(1, MAX_EPOCHS+1), d, marker="o", ms=4, lw=1.8,
                label=LABELS[k], color=COLORS[k])
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.set_xticks(range(1, MAX_EPOCHS+1)); ax.legend(fontsize=8, ncol=2)
    _save(fig, fname)

def bar_plot(vals, errs, title, ylabel, fname, fmt="%.2f", zoom=False):
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar([f"[{k}]" for k in CONFIG_ORDER], vals,
                  yerr=errs, capsize=5, color=[COLORS[k] for k in CONFIG_ORDER])
    ax.bar_label(bars, fmt=fmt, padding=3, fontsize=9)
    if zoom:
        lo, hi = min(vals), max(vals)
        ax.set_ylim(lo - max(2, 2*(hi-lo)), hi + max(1, (hi-lo)))
    ax.set_title(title); ax.set_ylabel(ylabel); ax.grid(axis="x", visible=False)
    _save(fig, fname)

stage_colors = plt.cm.viridis(np.linspace(0, 0.9, 5))

# 01 Training accuracy vs epochs
line_plot("train_acc",
          f"Training Accuracy vs Epoch ({DISTANCE_METRIC.capitalize()})",
          "Accuracy (%)", "01_train_accuracy_vs_epoch.png")

# 02 Training loss vs epochs
line_plot("train_loss",
          f"Training Loss vs Epoch ({DISTANCE_METRIC.capitalize()})",
          "Cross-Entropy Loss", "02_train_loss_vs_epoch.png")

# 03 Validation accuracy vs epochs
line_plot("val_acc",
          f"Validation Accuracy vs Epoch ({DISTANCE_METRIC.capitalize()})",
          "Accuracy (%)", "03_val_accuracy_vs_epoch.png")

# 04 Validation loss vs epochs
line_plot("val_loss",
          f"Validation Loss vs Epoch ({DISTANCE_METRIC.capitalize()})",
          "Cross-Entropy Loss", "04_val_loss_vs_epoch.png")

# 05 Final test accuracy (bar + CI)
bar_plot(
    [RESULTS[k]["test_acc"] for k in CONFIG_ORDER],
    [1.96*np.array(RESULTS[k]["per_episode_acc"]).std(ddof=1)/math.sqrt(N_TEST_TASKS) for k in CONFIG_ORDER],
    f"Test Accuracy 5-way 5-shot ({DISTANCE_METRIC.capitalize()}, 95% CI)",
    "Accuracy (%)", "05_test_accuracy_comparison.png", zoom=True)

# 06 Training time
bar_plot(
    [RESULTS[k]["train_time_total_s"]/60 for k in CONFIG_ORDER], None,
    "Total Training Time (10 epochs)", "Minutes",
    "06_training_time_comparison.png", fmt="%.1f min")

# 07 Inference time
bar_plot(
    [RESULTS[k]["infer_ms_per_episode"] for k in CONFIG_ORDER], None,
    "Inference Time per 5-way 5-shot Episode", "ms / episode",
    "07_inference_time_comparison.png", fmt="%.1f ms")

# 08 Peak GPU memory (grouped bar: train + infer)
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(CONFIG_ORDER)); w = 0.35
b1 = ax.bar(x-w/2, [RESULTS[k]["peak_train_gpu_mb"] for k in CONFIG_ORDER], w,
            label="Training", color="#4c78a8")
b2 = ax.bar(x+w/2, [RESULTS[k]["peak_infer_gpu_mb"] for k in CONFIG_ORDER], w,
            label="Inference", color="#f58518")
ax.bar_label(b1, fmt="%.0f", fontsize=8, padding=2)
ax.bar_label(b2, fmt="%.0f", fontsize=8, padding=2)
ax.set_xticks(x); ax.set_xticklabels([f"[{k}]" for k in CONFIG_ORDER])
ax.set_title("Peak GPU Memory (MB)"); ax.set_ylabel("MB"); ax.legend()
ax.grid(axis="x", visible=False)
_save(fig, "08_gpu_memory_comparison.png")

# 09 Stage weights over epochs (4 subplots)
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, key in zip(axes, CONFIG_ORDER):
    for i in range(5):
        init_v = STAGE_WEIGHTS[i]
        vals   = [r[f"stage_w{i+1}"] for r in RESULTS[key]["epoch_log"]]
        ax.plot([0]+list(range(1, MAX_EPOCHS+1)), [init_v]+vals,
                marker="o", ms=3, color=stage_colors[i], label=f"w{i+1} ({STAGE_NODES[i]})")
    ax.set_title(f"[{key}]"); ax.set_xlabel("Epoch")
axes[0].set_ylabel("Stage Weight"); axes[0].legend(fontsize=7)
fig.suptitle("Stage Weight Trajectories")
_save(fig, "09_stage_weight_trajectories.png")

# 10 Gamma trajectories (2 subplots: gamma1, gamma2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, col, ttl in zip(axes, ["gamma1", "gamma2"],
                         ["gamma1 (support attention)", "gamma2 (query attention)"]):
    for key in CONFIG_ORDER:
        vals = [r[col] for r in RESULTS[key]["epoch_log"]]
        ax.plot([0]+list(range(1, MAX_EPOCHS+1)), [gamma_init]+vals,
                marker="o", ms=3, label=f"[{key}]", color=COLORS[key])
    ax.set_title(ttl); ax.set_xlabel("Epoch")
axes[0].set_ylabel("Value"); axes[0].legend(fontsize=8)
fig.suptitle("Self-Attention Gamma Trajectories")
_save(fig, "10_gamma_trajectories.png")

# 11 GeM exponent over epochs (only for GeM configs)
gem_keys = [k for k in CONFIG_ORDER if EXPERIMENT_CONFIGS[k]["use_gem"]]
if gem_keys:
    fig, axes = plt.subplots(1, len(gem_keys), figsize=(6*len(gem_keys), 4), sharey=True)
    if len(gem_keys) == 1:
        axes = [axes]
    for ax, key in zip(axes, gem_keys):
        for i in range(5):
            col  = f"gem_p{i+1}"
            vals = [snap.get(col, float("nan")) for snap in RESULTS[key]["trajectory"]]
            epochs= [snap["epoch"] for snap in RESULTS[key]["trajectory"]]
            ax.plot(epochs, vals, marker="o", ms=3, color=stage_colors[i],
                    label=f"p{i+1} ({STAGE_NODES[i]})")
        ax.axhline(y=GEM_P_INIT, ls="--", color="gray", alpha=0.5, label="init")
        ax.set_title(f"[{key}]"); ax.set_xlabel("Epoch")
    axes[0].set_ylabel("GeM p"); axes[0].legend(fontsize=7)
    fig.suptitle("GeM Exponent Trajectories")
    _save(fig, "11_gem_p_trajectories.png")
else:
    print("  no GeM configs - skipping gem_p plot")

# 12 Per-episode accuracy boxplot
fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot([RESULTS[k]["per_episode_acc"] for k in CONFIG_ORDER],
                showmeans=True, patch_artist=True)
for patch, key in zip(bp["boxes"], CONFIG_ORDER):
    patch.set_facecolor(COLORS[key]); patch.set_alpha(0.6)
ax.set_xticks(range(1, len(CONFIG_ORDER)+1))
ax.set_xticklabels([f"[{k}]" for k in CONFIG_ORDER])
ax.set_title(f"Per-Episode Test Accuracy ({DISTANCE_METRIC.capitalize()}, same 600 episodes)")
ax.set_ylabel("Episode Accuracy (%)"); ax.grid(axis="x", visible=False)
_save(fig, "12_per_episode_accuracy_boxplot.png")

print(f"\nAll plots saved to {PLOT_DIR}")

In [ ]:
# ============================================================================
# Cell 13 — Experiment summary text + ZIP (no automatic download)
# ============================================================================
import zipfile, datetime

lines = []
lines.append("=" * 72)
lines.append("MSENet Stacked Experiment Summary")
lines.append(f"Generated: {datetime.datetime.now().isoformat()}")
lines.append("=" * 72)
lines.append("")
lines.append(f"DISTANCE_METRIC = '{DISTANCE_METRIC}'")
lines.append("  (change 'euclidean' to 'cosine' in Cell 3 to switch all experiments)")
lines.append("")
lines.append("Hyperparameters:")
for k, v in [
    ("Epochs x Episodes",       f"{MAX_EPOCHS} x {N_TRAINING_EPISODES}"),
    ("Evaluation protocol",     f"{TEST_WAY}-way {SHOT}-shot"),
    ("Val / Test tasks",        f"{N_VALIDATION_TASKS} / {N_TEST_TASKS}"),
    ("backbone_lr",             backbone_lr),
    ("weights_lr",              weights_lr),
    ("gem_lr",                  GEM_LR),
    ("EMA beta",                EMA_BETA),
    ("GeM P_INIT",              GEM_P_INIT),
    ("STAGE_WEIGHTS init",      STAGE_WEIGHTS),
    ("gamma_init",              gamma_init),
    ("COSINE_TEMP init",        COSINE_TEMP),
    ("AMP",                     USE_AMP),
    ("Seeds (train/val/test)",  f"{TRAIN_EPISODE_SEED}/{VAL_EPISODE_SEED}/{TEST_EPISODE_SEED}"),
]:
    lines.append(f"  {k:30s} = {v}")
lines.append("")
lines.append("Experiment Configurations:")
for key in CONFIG_ORDER:
    cfg = EXPERIMENT_CONFIGS[key]
    lines.append(f"  [{key}] {cfg['description']:25s}  "
                 f"ema={cfg['use_ema']}  gem={cfg['use_gem']}  layernorm={cfg['use_layernorm']}")
lines.append("")
lines.append("Final Results:")
hdr = f"  {'Exp':4s}  {'Description':25s}  {'Test%':>8s}  {'BestVal%':>9s}  {'BestEp':>6s}  {'TrainMin':>9s}"
lines.append(hdr)
for key in CONFIG_ORDER:
    r = RESULTS[key]
    lines.append(f"  [{key}]  {EXPERIMENT_CONFIGS[key]['description']:25s}  "
                 f"{r['test_acc']:7.2f}%  {r['best_val_acc']:8.2f}%  "
                 f"{r['best_epoch']:6d}  {r['train_time_total_s']/60:8.1f} min")
lines.append("")
lines.append("Environment:")
for k, v in ENV_INFO.items():
    lines.append(f"  {k}: {v}")

(RESULT_DIR / "experiment_summary.txt").write_text("\n".join(lines))
print("Saved: experiment_summary.txt")
print()
for line in lines:
    print(line)

# Verify no checkpoint files
bad = [p for p in RESULT_DIR.rglob("*") if p.suffix in (".pth", ".pt", ".ckpt", ".bin")]
assert not bad, f"Unexpected checkpoint files: {bad}"

# Create ZIP
zip_path = WORK_DIR / "msenet_stacked_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(RESULT_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, p.relative_to(RESULT_DIR))

print(f"\nFiles in ZIP ({zip_path.stat().st_size/1e6:.2f} MB):")
for p in sorted(RESULT_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(RESULT_DIR)}  ({p.stat().st_size/1024:.1f} KB)")
print(f"\nZIP saved at: {zip_path}")
print("Download it from Kaggle's Output / Files panel (no auto-download).")